# Tema 01: Introducción a Colab y Pandas

**Taller:** Análisis y Visualización Interactiva en Python
**Duración estimada de esta sesión:** 30 minutos
**Herramienta principal:** Google Colab + Pandas
**Modalidad de práctica:** Google Colab

---

> 📌 **Nota para el profesor:** esta notebook está diseñada para proyectarse y ejecutarse en vivo. Las secciones marcadas como **Práctica guiada** se resuelven junto con el grupo; las marcadas como **Práctica independiente** las resuelven los participantes en sus propias copias de la notebook (`Archivo → Guardar una copia en Drive`).

## 🎯 Objetivos de aprendizaje

Al finalizar este tema, el participante será capaz de:

- Reconocer los elementos principales del entorno de Google Colab (celdas, entorno de ejecución, integración con Drive/GitHub).
- Cargar un dataset CSV en un DataFrame de Pandas y realizar una inspección inicial de su contenido.
- Seleccionar, filtrar y ordenar datos usando `loc`, `iloc` e indexado booleano.
- Construir resúmenes agregados con `groupby` como base para el resto del taller.

## 🧠 Contenido teórico

### 1. ¿Qué es Google Colab?

Google Colaboratory (**Colab**) es un entorno de notebooks Jupyter que se ejecuta completamente en la nube de Google. No requiere instalación: cada notebook corre en una máquina virtual temporal ("entorno de ejecución" o *runtime*) a la que puedes conectarte con un clic.

Puntos clave para explicar en clase:

- **Celdas de código y de texto (Markdown):** una notebook combina explicaciones, ecuaciones, imágenes y código ejecutable en un mismo documento — ideal para análisis reproducible y "storytelling" con datos.
- **Entorno de ejecución (runtime):** cada vez que abres una notebook y ejecutas una celda, Colab te asigna una máquina con Python y muchas librerías preinstaladas (Pandas, NumPy, Matplotlib, etc.). El estado (variables, DataFrames) se pierde si el entorno se reinicia o desconecta.
- **Orden de ejecución:** las celdas *no* se ejecutan automáticamente en orden de arriba hacia abajo; se ejecutan en el orden en que tú las corres. El número entre corchetes `[ ]` indica ese orden — una fuente común de errores para principiantes.
- **Integración con Drive y GitHub:** puedes guardar copias en tu Google Drive (`Archivo → Guardar una copia en Drive`) o en un repositorio de GitHub (`Archivo → Guardar una copia en GitHub`), y cargar archivos desde ambos.
- **Aceleradores de hardware:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución` permite habilitar GPU/TPU (no se usan en este taller, pero es útil mencionarlo).

**Atajos útiles:**

| Acción | Atajo |
|---|---|
| Ejecutar celda y avanzar | `Shift + Enter` |
| Ejecutar celda y quedarse en ella | `Ctrl/Cmd + Enter` |
| Insertar celda de código abajo | `Ctrl/Cmd + M B` |
| Convertir celda a Markdown | `Ctrl/Cmd + M M` |
| Reiniciar entorno de ejecución | Menú `Entorno de ejecución → Reiniciar entorno de ejecución` |

### 2. ¿Qué es Pandas y por qué es el punto de partida?

**Pandas** es la librería estándar de Python para manipular datos tabulares (filas y columnas, como una hoja de cálculo o una tabla SQL). Sus dos estructuras fundamentales son:

- **`Series`**: un arreglo unidimensional etiquetado (una sola columna).
- **`DataFrame`**: una tabla bidimensional, colección de `Series` que comparten el mismo índice — el objeto con el que trabajaremos casi todo el taller.

Casi todas las herramientas de visualización que veremos (Matplotlib, Seaborn, Plotly, Dash, Bokeh, Folium) reciben datos en forma de `DataFrame` o estructuras derivadas de él. Dominar Pandas es el cimiento de todo lo demás.

### 3. Operaciones esenciales que veremos hoy

- **Carga de datos:** `pd.read_csv()`.
- **Inspección:** `.head()`, `.tail()`, `.shape`, `.dtypes`, `.info()`, `.describe()`.
- **Selección:** indexado con corchetes `df["col"]`, `.loc[filas, columnas]` (por etiqueta) y `.iloc[filas, columnas]` (por posición).
- **Filtrado booleano:** `df[df["columna"] > valor]`.
- **Agregación:** `.groupby("columna").agg(...)`, `.sort_values()`.

En los siguientes temas usaremos estas mismas operaciones como base antes de graficar: *todo buen gráfico empieza con un buen DataFrame*.

## ⚙️ Configuración del entorno

Ejecuta la siguiente celda para cargar el dataset `tema01_ventas_tienda.csv` (ventas sintéticas de una tienda con productos, categorías, regiones y canal de venta durante 2024).

In [ ]:
# === Carga del dataset ===
# Opción 1 (recomendada una vez publicado el repositorio del taller):
# reemplaza <usuario>/<repositorio> por la ruta real de tu repo de GitHub
# y ejecuta esta celda. Usa el botón "Raw" de GitHub para obtener la URL.
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/<usuario>/<repositorio>/main/"
    "datasets/tema01_ventas_tienda.csv"
)

import pandas as pd

try:
    df = pd.read_csv(GITHUB_RAW_URL)
    print("Datos cargados desde GitHub ✅  ->", df.shape)
except Exception as e:
    print("No se pudo leer desde GitHub todavía (repo no configurado o sin internet).")
    print("Sube manualmente el archivo 'tema01_ventas_tienda.csv' cuando se te solicite.")
    try:
        from google.colab import files
        subido = files.upload()  # selecciona tema01_ventas_tienda.csv
        df = pd.read_csv(list(subido.keys())[0])
    except ImportError:
        # Fuera de Colab (por ejemplo, ejecución local de prueba):
        df = pd.read_csv("tema01_ventas_tienda.csv")

df.head()

## 🧭 Práctica guiada

El profesor ejecuta y explica cada celda en vivo. Los participantes ejecutan las mismas celdas en su copia y observan la salida.

### Paso 1 · Inspección inicial

Antes de graficar cualquier cosa, siempre inspeccionamos la forma y el tipo de datos.

In [ ]:
print("Dimensiones (filas, columnas):", df.shape)
print("\nTipos de dato por columna:")
print(df.dtypes)
df.info()

**Salida esperada:** verás que `df` tiene 500 filas y 8 columnas; `fecha` puede cargarse como texto (`object`) — lo convertiremos a fecha a continuación.

In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"])
df.describe(include="all").T

### Paso 2 · Selección con `loc` / `iloc`

In [ ]:
# .iloc -> selección por POSICIÓN (como una matriz)
print(df.iloc[0:3, 0:4])  # primeras 3 filas, primeras 4 columnas

# .loc -> selección por ETIQUETA (nombre de columna / índice)
df.loc[0:2, ["producto", "categoria", "venta_total"]]

### Paso 3 · Filtrado booleano

Queremos las ventas de la región **Centro** con un `venta_total` mayor a 1000.

In [ ]:
filtro = (df["region"] == "Centro") & (df["venta_total"] > 1000)
ventas_centro_altas = df[filtro]
print(f"Encontramos {len(ventas_centro_altas)} ventas que cumplen la condición.")
ventas_centro_altas.sort_values("venta_total", ascending=False).head()

### Paso 4 · Agregación con `groupby`

¿Qué categoría genera más ingresos? ¿Y qué región vende más unidades?

In [ ]:
ingresos_por_categoria = (
    df.groupby("categoria")["venta_total"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)
print("Ingresos totales por categoría:")
print(ingresos_por_categoria)

resumen_region = (
    df.groupby("region")
    .agg(unidades_vendidas=("cantidad", "sum"),
         ingreso_total=("venta_total", "sum"),
         ticket_promedio=("venta_total", "mean"))
    .round(2)
    .sort_values("ingreso_total", ascending=False)
)
resumen_region

## ✍️ Práctica independiente

Resuelve los siguientes ejercicios en celdas nuevas. Usa el DataFrame `df` ya cargado. Al final de la notebook encontrarás una sección de **soluciones** — intenta resolverlos antes de consultarla.

**Ejercicio 1.** Calcula el total de unidades (`cantidad`) vendidas por `canal_venta`.

In [ ]:
# TODO: tu código aquí

**Ejercicio 2.** Encuentra la fila (venta individual) con el `venta_total` más alto de todo el dataset. Pista: `.sort_values()` o `.idxmax()`.

In [ ]:
# TODO: tu código aquí

**Ejercicio 3.** Crea una nueva columna booleana `venta_alta` que sea `True` cuando `venta_total > 1500`. Cuenta cuántas ventas son altas con `.value_counts()`.

In [ ]:
# TODO: tu código aquí

**Ejercicio 4 (reto).** Construye una tabla dinámica (`pd.pivot_table`) con el precio unitario **promedio** por `categoria` (filas) y `canal_venta` (columnas).

In [ ]:
# TODO: tu código aquí

---
### ✅ Soluciones (referencia para el profesor)

Ejecuta esta celda solo después de intentar los ejercicios.

In [ ]:
# Ejercicio 1
sol1 = df.groupby("canal_venta")["cantidad"].sum()
print("Ejercicio 1:\n", sol1, "\n")

# Ejercicio 2
sol2 = df.loc[df["venta_total"].idxmax()]
print("Ejercicio 2:\n", sol2, "\n")

# Ejercicio 3
df["venta_alta"] = df["venta_total"] > 1500
sol3 = df["venta_alta"].value_counts()
print("Ejercicio 3:\n", sol3, "\n")

# Ejercicio 4
sol4 = pd.pivot_table(df, values="precio_unitario", index="categoria",
                       columns="canal_venta", aggfunc="mean").round(2)
print("Ejercicio 4:\n", sol4)

## 🔎 Cierre y puente al siguiente tema

Con Pandas ya sabemos **explorar, limpiar, filtrar y resumir** datos. En el **Tema 02 (Matplotlib)** tomaremos resúmenes como los de `groupby` de esta sesión y los convertiremos en gráficos estáticos (líneas, barras) — la base de toda visualización en Python.

## 📚 Recursos adicionales

- [Documentación oficial de Pandas](https://pandas.pydata.org/docs/)
- [10 minutes to pandas (guía rápida oficial)](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Pandas Cheat Sheet oficial (PDF)](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
- [Preguntas frecuentes de Google Colab](https://research.google.com/colaboratory/faq.html)
- [Guía de atajos de teclado de Colab](https://colab.research.google.com/notebooks/basic_features_overview.ipynb)